## Colab Setup

In [1]:

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/ImmunoPath"
DATA_DIR = f"{PROJECT_DIR}/data"
TRAINING_DIR = f"{DATA_DIR}/training"
MODEL_DIR = f"{PROJECT_DIR}/models/immunopath-v1"
RESULTS_DIR = f"{PROJECT_DIR}/results/phase5"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

!pip install -q --no-cache-dir \
    "transformers>=4.50.0" \
    "accelerate>=0.34.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.44.0"

try:
    !pip install -q flash-attn --no-build-isolation
    FLASH_ATTN_AVAILABLE = True
    print("Flash Attention 2 installed")
except:
    FLASH_ATTN_AVAILABLE = False
    print("Flash Attention not available")

from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in to HuggingFace")
except:
    print("Set HF_TOKEN in Colab Secrets")

import torch
import gc

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected.")

import transformers, accelerate, bitsandbytes
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 257.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 70.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Flash Attention 2 installed
Logged in to HuggingFace
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
transformers: 5.0.0
accelerate: 1.12.0
bitsandbytes: 0.49.2


## Configuration

In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    model_id: str = "google/medgemma-1.5-4b-it"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    use_dora: bool = True
    target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )
    num_epochs: int = 3
    batch_size: int = 1
    grad_accum: int = 8
    lr: float = 1e-4
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_patches: int = 4
    max_length: int = 1536
    train_path: str = f"{TRAINING_DIR}/train.jsonl"
    val_path: str = f"{TRAINING_DIR}/val.jsonl"
    output_dir: str = MODEL_DIR
    log_dir: str = RESULTS_DIR
    logging_steps: int = 10
    eval_steps: int = 100
    save_steps: int = 100

cfg = Config()
print(f"Config: DoRA r={cfg.lora_r}, alpha={cfg.lora_alpha}, "
      f"batch={cfg.batch_size}x{cfg.grad_accum}={cfg.batch_size*cfg.grad_accum}, "
      f"lr={cfg.lr}, epochs={cfg.num_epochs}, "
      f"max_patches={cfg.max_patches}, max_length={cfg.max_length}")


Config: DoRA r=16, alpha=32, batch=1x8=8, lr=0.0001, epochs=3, max_patches=4, max_length=1536


In [3]:
import json, shutil, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

LOCAL_TRAIN_DIR = "/content/immunopath_local"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

print("Pre-copying training data to local SSD for faster I/O...")

# Always overwrite JSONL files (small, fast)
for split in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    src = f"{TRAINING_DIR}/{split}"
    dst = f"{LOCAL_TRAIN_DIR}/{split}"
    if os.path.exists(src):
        shutil.copy2(src, dst)

# Collect ONLY the patch files actually referenced in train+val
DRIVE_PREFIX = "/content/drive/MyDrive/ImmunoPath"
patch_files = set()
for split in ["train.jsonl", "val.jsonl"]:
    src = f"{TRAINING_DIR}/{split}"
    if not os.path.exists(src):
        continue
    with open(src) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample.get("patch_paths", [])[:cfg.max_patches]:
                assert p.startswith(DRIVE_PREFIX), f"Unexpected path prefix: {p}"
                patch_files.add(p)

# Pre-create all destination directories in one pass (avoids per-file makedirs)
dst_dirs = set()
for src_file in patch_files:
    dst_dirs.add(os.path.dirname(src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)))
for d in dst_dirs:
    os.makedirs(d, exist_ok=True)

# Filter to only files that need copying (skip if already on local SSD)
to_copy = []
for src_file in patch_files:
    dst_file = src_file.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)
    if not os.path.exists(dst_file):
        to_copy.append((src_file, dst_file))

print(f"Need to copy {len(to_copy)}/{len(patch_files)} patch files "
      f"({len(patch_files) - len(to_copy)} already cached)")

# Parallel copy — 16 threads saturate GDrive FUSE bandwidth
def _copy_one(pair):
    shutil.copy2(pair[0], pair[1])

if to_copy:
    from tqdm import tqdm
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(tqdm(ex.map(_copy_one, to_copy), total=len(to_copy), desc="Copying patches"))

# Rewrite JSONL paths to point to local SSD
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    updated = []
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            sample["patch_paths"] = [
                p.replace(DRIVE_PREFIX, LOCAL_TRAIN_DIR)
                for p in sample["patch_paths"]
            ]
            updated.append(json.dumps(sample))
    with open(local_path, "w") as f:
        f.write("\n".join(updated) + "\n")

# Validate: check ALL referenced patches exist locally
missing = []
for split in ["train.jsonl", "val.jsonl"]:
    local_path = f"{LOCAL_TRAIN_DIR}/{split}"
    if not os.path.exists(local_path):
        continue
    with open(local_path) as f:
        for line in f:
            sample = json.loads(line.strip())
            for p in sample["patch_paths"][:cfg.max_patches]:
                if not os.path.exists(p):
                    missing.append(p)
assert not missing, f"FATAL: {len(missing)} patch files missing! Example: {missing[0]}"

cfg.train_path = f"{LOCAL_TRAIN_DIR}/train.jsonl"
cfg.val_path = f"{LOCAL_TRAIN_DIR}/val.jsonl"
print(f"All patches verified. Training paths: {LOCAL_TRAIN_DIR}")
print("SSD pre-copy complete — GDrive I/O eliminated during training")

Pre-copying training data to local SSD for faster I/O...
Need to copy 3375/3375 patch files (0 already cached)


Copying patches: 100%|██████████| 3375/3375 [01:11<00:00, 47.27it/s]


All patches verified. Training paths: /content/immunopath_local
SSD pre-copy complete — GDrive I/O eliminated during training


## Load Model + Apply DoRA Adapters

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

print(f"Loading {cfg.model_id}...")

processor = AutoProcessor.from_pretrained(cfg.model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

attn_impl = "flash_attention_2" if FLASH_ATTN_AVAILABLE else "eager"

model = AutoModelForImageTextToText.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation=attn_impl,
    quantization_config=bnb_config,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=list(cfg.target_modules),
    use_dora=cfg.use_dora,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"Model loaded + DoRA applied ({attn_impl})")
print(f"VRAM allocated: {allocated:.2f} GB, reserved: {reserved:.2f} GB")


Loading google/medgemma-1.5-4b-it...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

trainable params: 33,891,456 || all params: 4,333,970,928 || trainable%: 0.7820
Model loaded + DoRA applied (flash_attention_2)
VRAM allocated: 4.73 GB, reserved: 10.69 GB


## ImmunoPathDataset  (response-only loss masking)

In [5]:
import json
import torch
from PIL import Image
from torch.utils.data import Dataset
from typing import Dict, List, Any

# Pre-compute the model-turn marker tokens once (used to find prompt/response boundary)
# This avoids calling apply_chat_template twice per sample (which would double image processing)
_MODEL_TURN_MARKER = None  # Set after processor is loaded

class ImmunoPathDataset(Dataset):

    def __init__(self, data_path, processor, max_patches=4, max_length=1536):
        global _MODEL_TURN_MARKER
        self.processor = processor
        self.max_patches = max_patches
        self.max_length = max_length

        # Pre-compute model turn marker tokens once
        if _MODEL_TURN_MARKER is None:
            _MODEL_TURN_MARKER = processor.tokenizer.encode(
                "<start_of_turn>model\n", add_special_tokens=False
            )

        self.samples = []
        with open(data_path) as f:
            for line in f:
                self.samples.append(json.loads(line.strip()))
        print(f"Loaded {len(self.samples)} samples from {data_path}")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _load_image(path):
        try:
            img = Image.open(path).convert("RGB")
            # Do NOT resize here — the processor handles native resize internally.
            # Manual 336×336 resize before processor causes double-resize (quality loss + wasted compute).
            return img
        except Exception:
            return None

    def _load_images(self, paths):
        paths = paths[:self.max_patches]
        # For ≤4 images, sequential load is faster than ThreadPoolExecutor overhead
        images = []
        for p in paths:
            img = self._load_image(p)
            if img is not None:
                images.append(img)
        if not images:
            images = [Image.new("RGB", (512, 512), "white")]
        return images

    def __getitem__(self, idx):
        sample = self.samples[idx]
        images = self._load_images(sample["patch_paths"])

        prompt_text = sample["prompt"]
        response_text = sample["response"]

        user_content = [{"type": "image", "image": img} for img in images]
        user_content.append({"type": "text", "text": prompt_text})

        full_messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": [{"type": "text", "text": response_text}]},
        ]

        # Single apply_chat_template call (processes images only ONCE)
        full_inputs = self.processor.apply_chat_template(
            full_messages,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )

        if "token_type_ids" not in full_inputs:
            full_inputs["token_type_ids"] = torch.zeros_like(full_inputs["input_ids"])

        input_ids = full_inputs["input_ids"].squeeze(0)
        attention_mask = full_inputs["attention_mask"].squeeze(0)
        token_type_ids = full_inputs["token_type_ids"].squeeze(0)

        # Find prompt/response boundary by locating "<start_of_turn>model\n" marker
        # This replaces the second apply_chat_template call (eliminates 2× image processing)
        input_ids_list = input_ids.tolist()
        marker = _MODEL_TURN_MARKER
        prompt_len = 0
        for start_idx in range(len(input_ids_list) - len(marker), -1, -1):
            if input_ids_list[start_idx:start_idx + len(marker)] == marker:
                prompt_len = start_idx + len(marker)
                break

        labels = input_ids.clone()
        labels[:prompt_len] = -100

        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        # Close PIL images to free memory
        for img in images:
            img.close()

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "labels": labels,
        }

        for k, v in full_inputs.items():
            if torch.is_tensor(v) and k not in out:
                out[k] = v.squeeze(0)

        return out


print("Building train dataset...")
train_dataset = ImmunoPathDataset(cfg.train_path, processor, cfg.max_patches, cfg.max_length)

print("Building val dataset...")
val_dataset = ImmunoPathDataset(cfg.val_path, processor, cfg.max_patches, cfg.max_length)

if len(train_dataset) > 0:
    sample = train_dataset[0]
    print("\nSample shapes:")
    for k, v in sample.items():
        if torch.is_tensor(v):
            print(f"  {k}: {v.shape} ({v.dtype})")
    n_masked = (sample["labels"] == -100).sum().item()
    n_total = sample["labels"].shape[0]
    print(f"  Masked tokens: {n_masked}/{n_total}")
    del sample
    gc.collect()

Building train dataset...
Loaded 751 samples from /content/immunopath_local/train.jsonl
Building val dataset...
Loaded 94 samples from /content/immunopath_local/val.jsonl

Sample shapes:
  input_ids: torch.Size([1462]) (torch.int64)
  attention_mask: torch.Size([1462]) (torch.int64)
  token_type_ids: torch.Size([1462]) (torch.int64)
  labels: torch.Size([1462]) (torch.int64)
  pixel_values: torch.Size([4, 3, 896, 896]) (torch.float32)
  Masked tokens: 1304/1462


## Custom Data Collator (variable-length padding)

In [15]:
from dataclasses import dataclass as dc
from typing import List, Dict, Any
import torch


@dc
class ImmunoPathCollator:
    processor: Any
    padding: str = "longest"

    def __call__(self, features):
        label_list = [f.pop("labels") for f in features]

        pixel_values_list = None
        if "pixel_values" in features[0]:
            pixel_values_list = [f.pop("pixel_values") for f in features]

        pad_id = self.processor.tokenizer.pad_token_id or 0
        max_len = max(f["input_ids"].shape[0] for f in features)

        batch = {}

        for key in ["input_ids", "attention_mask", "token_type_ids"]:
            tensors = [f[key] for f in features]
            padded = []
            for t in tensors:
                pad_size = max_len - t.shape[0]
                if pad_size > 0:
                    if key == "attention_mask":
                        pad_tensor = torch.zeros(pad_size, dtype=t.dtype)
                    else:
                        pad_tensor = torch.full((pad_size,), pad_id, dtype=t.dtype)
                    t = torch.cat([t, pad_tensor], dim=0)
                padded.append(t)
            batch[key] = torch.stack(padded)

        padded_labels = []
        for lb in label_list:
            pad_size = max_len - lb.shape[0]
            if pad_size > 0:
                pad_tensor = torch.full((pad_size,), -100, dtype=lb.dtype)
                lb = torch.cat([lb, pad_tensor], dim=0)
            padded_labels.append(lb)
        batch["labels"] = torch.stack(padded_labels)

        if pixel_values_list is not None:
          pixel_values_list = [pv.unsqueeze(0) if pv.dim() == 3 else pv for pv in pixel_values_list]
          batch["pixel_values"] = torch.cat(pixel_values_list, dim=0)

        return batch


collator = ImmunoPathCollator(processor=processor)
print("Data collator ready")


Data collator ready


## Training Arguments + Trainer

In [16]:
from transformers import TrainingArguments, Trainer
import json, time

ckpt_dir = f"{cfg.output_dir}/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

training_args = TrainingArguments(
    output_dir=ckpt_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=cfg.logging_steps,
    eval_strategy="steps",
    eval_steps=cfg.eval_steps,
    save_strategy="steps",
    save_steps=cfg.save_steps,
    save_total_limit=2,
    bf16=True,
    optim="adamw_bnb_8bit",
    max_grad_norm=cfg.max_grad_norm,
    report_to="tensorboard",
    logging_dir=f"{cfg.log_dir}/tb_logs",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print(f"Trainer ready (eval every {cfg.eval_steps} steps)")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Effective batch size: {cfg.batch_size * cfg.grad_accum}")
est_steps = (len(train_dataset) // (cfg.batch_size * cfg.grad_accum)) * cfg.num_epochs
print(f"Estimated steps: ~{est_steps}")

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"VRAM before training: allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready (eval every 100 steps)
Train samples: 751
Val samples:   94
Effective batch size: 8
Estimated steps: ~279
VRAM before training: allocated=5.24 GB, reserved=9.22 GB


## TRAIN!!

In [13]:
import shutil, os
ckpt_dir = "/content/drive/MyDrive/ImmunoPath/models/immunopath-v1/checkpoints"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"Deleted {ckpt_dir}")
else:
    print("No checkpoints found")

Deleted /content/drive/MyDrive/ImmunoPath/models/immunopath-v1/checkpoints


In [17]:
print("=" * 60)
print("STARTING FINE-TUNING")
print("=" * 60)

start_time = time.time()

resume_from = None
if os.path.exists(ckpt_dir):
    ckpts = [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")]
    if ckpts:
        latest = sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1]
        resume_from = f"{ckpt_dir}/{latest}"
        print(f"Resuming from {resume_from}")

train_result = trainer.train(resume_from_checkpoint=resume_from)

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {train_result.training_loss:.4f}")


STARTING FINE-TUNING


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
100,0.092190,0.094561
200,0.089972,0.092220



Training complete in 96.3 minutes
Final train loss: 0.0925


## Save DoRA Adapters + Processor

In [19]:
# Save the PEFT adapters (small — just DoRA weights)
adapter_dir = f"{cfg.output_dir}/lora_adapters"
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)

# Verify saved files
saved_files = os.listdir(adapter_dir)
print(f"DoRA adapters saved to {adapter_dir}")
print(f"   Files: {saved_files}")

# Also save via trainer (includes training state)
trainer.save_model(cfg.output_dir)
processor.save_pretrained(cfg.output_dir)
print(f"Full model checkpoint saved to {cfg.output_dir}")

DoRA adapters saved to /content/drive/MyDrive/ImmunoPath/models/immunopath-v1/lora_adapters
   Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
Full model checkpoint saved to /content/drive/MyDrive/ImmunoPath/models/immunopath-v1


## Save Training Log + Metrics

In [20]:
from datetime import datetime

# Extract training history
log_history = trainer.state.log_history

# Separate train and eval logs
train_logs = [l for l in log_history if "loss" in l and "eval_loss" not in l]
eval_logs = [l for l in log_history if "eval_loss" in l]

training_report = {
    "phase": 5,
    "timestamp": datetime.now().isoformat(),
    "model_id": cfg.model_id,
    "peft_method": "DoRA" if cfg.use_dora else "LoRA",
    "peft_config": {
        "r": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": list(cfg.target_modules),
        "use_dora": cfg.use_dora,
    },
    "training_config": {
        "batch_size": cfg.batch_size,
        "grad_accum": cfg.grad_accum,
        "effective_batch": cfg.batch_size * cfg.grad_accum,
        "lr": cfg.lr,
        "epochs": cfg.num_epochs,
        "bf16": True,
        "quantised": True,
    },
    "data": {
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
    },
    "results": {
        "final_train_loss": train_result.training_loss,
        "best_eval_loss": min((l["eval_loss"] for l in eval_logs), default=None),
        "training_time_minutes": round(elapsed / 60, 1),
    },
    "train_logs": train_logs,
    "eval_logs": eval_logs,
}

log_path = f"{cfg.log_dir}/training_log.json"
with open(log_path, 'w') as f:
    json.dump(training_report, f, indent=2, default=str)
print(f"Training log saved to {log_path}")


Training log saved to /content/drive/MyDrive/ImmunoPath/results/phase5/training_log.json


## Quick Inference Test (verify fine-tuned model works)

In [21]:
from PIL import Image

# Load a val sample
val_sample = val_dataset.samples[0] if val_dataset.samples else None

if val_sample:
    # Match inference patch count to training (cfg.max_patches=4)
    images = [Image.open(p).convert("RGB") for p in val_sample["patch_paths"][:cfg.max_patches]]
    if not images:
        images = [Image.new("RGB", (512, 512), "white")]

    prompt = val_sample["prompt"]

    # Build messages
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {
        k: v.to(model.device, dtype=torch.bfloat16) if v.is_floating_point()
        else v.to(model.device)
        for k, v in inputs.items()
        if torch.is_tensor(v)
    }
    input_len = inputs["input_ids"].shape[-1]

    model.eval()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False,
            use_cache=True,
        )

    generated = output_ids[0][input_len:]
    response_text = processor.decode(generated, skip_special_tokens=True).strip()

    # Close images
    for img in images:
        img.close()

    print("Fine-tuned model inference test:")
    print(f"   Output preview: {response_text[:300]}")

    # Try to parse JSON
    try:
        clean = response_text
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start != -1 and end > start:
            parsed = json.loads(clean[start:end])
            print(f"   Valid JSON with keys: {list(parsed.keys())}")
        else:
            print("   No JSON object found in output")
    except json.JSONDecodeError as e:
        print(f"   JSON parse error: {e}")

    # Ground truth comparison
    print(f"\n   Ground truth (first 200 chars):")
    print(f"   {val_sample['response'][:200]}")
else:
    print("No val samples available for inference test")

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Fine-tuned model inference test:
   Output preview: {
  "cd274_expression": "high",
  "cd274_note": "RNA proxy (TCGA RNA-seq); NOT PD-L1 IHC TPS. Confirm with IHC for treatment-critical decisions.",
  "msi_status": "MSS",
  "tme_subtype": "IE",
  "til_fraction": 0.701,
  "til_density": "high",
  "immune_phenotype": "inflamed",
  "cd8_infiltration": "
   Valid JSON with keys: ['cd274_expression', 'cd274_note', 'msi_status', 'tme_subtype', 'til_fraction', 'til_density', 'immune_phenotype', 'cd8_infiltration', 'immune_score', 'prediction_entropy', 'low_confidence_flag']

   Ground truth (first 200 chars):
   {
  "cd274_expression": "high",
  "cd274_note": "RNA proxy (TCGA RNA-seq); NOT PD-L1 IHC TPS. Confirm with IHC for treatment-critical decisions.",
  "msi_status": "MSS",
  "tme_subtype": "F",
  "til_f


##  Phase 5 Summary

In [22]:
print("\n" + "=" * 60)
print("PHASE 5 — FINE-TUNING COMPLETE")
print("=" * 60)
print(f"\n  Model:       {cfg.model_id}")
print(f"  PEFT:        {'DoRA' if cfg.use_dora else 'LoRA'} (r={cfg.lora_r}, α={cfg.lora_alpha})")
print(f"  Quantised:   {'QDoRA (4-bit)' if True else 'No (full bf16)'}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples:   {len(val_dataset)}")
print(f"  Final loss:    {train_result.training_loss:.4f}")
best_eval = min((l["eval_loss"] for l in eval_logs), default="N/A")
print(f"  Best eval loss: {best_eval}")
print(f"  Training time: {elapsed/60:.1f} min")
print(f"\n  Adapters saved: {adapter_dir}")
print(f"  Log saved:      {log_path}")

print(f"\nUPDATE PHASE_TRACKER.md:")
print(f"  Phase 5 Status: DONE")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  Best eval loss:   {best_eval}")
print(f"  Training time:    {elapsed/60:.1f} min")
print(f"  Adapters:         {adapter_dir}")

print(f"\n{'=' * 60}")
print("NEXT: Phase 6 — Evaluation + Calibration")
print(f"{'=' * 60}")


PHASE 5 — FINE-TUNING COMPLETE

  Model:       google/medgemma-1.5-4b-it
  PEFT:        DoRA (r=16, α=32)
  Quantised:   QDoRA (4-bit)
  Train samples: 751
  Val samples:   94
  Final loss:    0.0925
  Best eval loss: 0.09221994131803513
  Training time: 96.3 min

  Adapters saved: /content/drive/MyDrive/ImmunoPath/models/immunopath-v1/lora_adapters
  Log saved:      /content/drive/MyDrive/ImmunoPath/results/phase5/training_log.json

UPDATE PHASE_TRACKER.md:
  Phase 5 Status: DONE
  Final train loss: 0.0925
  Best eval loss:   0.09221994131803513
  Training time:    96.3 min
  Adapters:         /content/drive/MyDrive/ImmunoPath/models/immunopath-v1/lora_adapters

NEXT: Phase 6 — Evaluation + Calibration
